<a href="https://colab.research.google.com/github/TienNguyen0712/hybrid-llm-tabular-pipeline-for-icu-mortality-prediction/blob/main/notebooks/predict_mortality_using_mimic_iv_baseline_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Xây dựng mô hình dự đoán tỷ lệ tử vong của ICU từ MIMIC-IV**

**Mục tiêu:**

Notebook này thực hiện xây dựng một mô hình dự đoán tỷ lệ tử vong từ bảng MIMIC bằng các mô hình học máy. Nhằm tạo ra một chuẩn dữ liệu để so sánh với các kỹ thuật khác trong tương lai (nếu có)

**Bộ dữ liệu**

MIMIC-IV là cơ sở dữ liệu lớn gồm các bảng về thông tin bệnh nhân, lâm sàng, các thủ thuật, v...

Lý do chọn bộ dữ liệu này do tính phức tạp, sát với dữ liệu thực tế, việc xử lý dữ liệu này là một trong những bước khó khắn, ...

- 2 module chính được sử dụng trong notebook này chính là `hosp` và `icu`. Chi tiết các bảng lựa chọn sẽ được mô tả ở dưới


## **1. Nạp thư viện và dữ liệu cần thiết**

In [9]:
# Bỏ comment nếu chạy trên Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

import os
from typing import List, Tuple, Dict, Optional

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")


✓ Libraries loaded


In [2]:
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq

def convert_csv_to_parquet_batch(
    table_configs,
    input_dir,
    output_dir,
    folder,
    block_size=100_000_000,
    compression='snappy'
):
    """
    Chuyển đổi danh sách các bảng csv.gz sang Parquet phân theo module.
    """
    # Ghép folder vào đường dẫn output_dir
    target_output_dir = os.path.join(output_dir, folder)
    target_input_dir = os.path.join(input_dir, folder)

    os.makedirs(target_output_dir, exist_ok=True)
    read_options = pv.ReadOptions(block_size=block_size)

    print(f"ĐANG XỬ LÝ MODULE: [{folder.upper()}]")


    for filename, columns in table_configs.items():
        input_file_path = os.path.join(target_input_dir, filename)

        # Lưu kết quả vào đúng thư mục con của module đó
        output_filename = filename.replace('.csv.gz', '.parquet').replace('.csv', '.parquet')
        output_file_path = os.path.join(target_output_dir, output_filename)

        print(f"\n- Đang xử lý: {filename}")
        print(f"  + Nguồn: {input_file_path}")
        print(f"  + Đích:  {output_file_path}")

        if not os.path.exists(input_file_path):
            print(f"Cảnh báo: Không tìm thấy file '{input_file_path}'. Đang bỏ qua...")
            continue

        convert_options = pv.ConvertOptions(include_columns=columns) if columns else pv.ConvertOptions()

        try:
            reader = pv.open_csv(
                input_file_path,
                read_options=read_options,
                convert_options=convert_options
            )

            writer = None
            for i, batch in enumerate(reader):
                table = pa.Table.from_batches([batch])

                if writer is None:
                    writer = pq.ParquetWriter(output_file_path, table.schema, compression=compression)

                writer.write_table(table)
                print(f"   -> [{filename}] Đã ghi batch {i + 1}...")

            if writer:
                writer.close()
                print(f"Hoàn thành: {output_filename}")
            else:
                print(f"File rỗng hoặc không có dữ liệu.")

        except Exception as e:
            print(f"Lỗi trong quá trình xử lý file {filename}: {e}")

    print(f"\nĐã hoàn thành toàn bộ bảng trong module [{folder.upper()}]!")

In [3]:
# Đường dẫn gốc đến dữ liệu nguồn và dữ liệu đích
BASE_INPUT_DIR = '/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/NCKH-DDU1231/outputs_parquet'

BLOCK_SIZE = 100_000_000  # 100MB cho mỗi batch đọc
COMPRESSION = 'snappy'

# ĐỊNH NGHĨA CÁC MODULE VÀ BẢNG TƯƠNG ỨNG
MODULE_CONFIGS = {
    'icu': {
        'chartevents.csv.gz': [
            'subject_id', 'hadm_id', 'stay_id',
            'charttime', 'itemid', 'valuenum',
            'value', 'valueuom'
        ],

        'inputevents.csv.gz': None,
        'outputevents.csv.gz': None,
        'procedureevents.csv.gz': None,
        'icustays.csv.gz': None,
    },

    'hosp': {
        'patients.csv.gz': [
            'subject_id', 'gender', 'anchor_age'
        ],

        'admissions.csv.gz': None,

        'labevents.csv.gz': [
            'subject_id', 'hadm_id',
            'charttime', 'itemid', 'valuenum',
            'value', 'valueuom'
        ]
    }
}

In [6]:
for folder_name, table_configs in MODULE_CONFIGS.items():
        convert_csv_to_parquet_batch(
            table_configs=table_configs,
            input_dir=BASE_INPUT_DIR,
            output_dir=BASE_OUTPUT_DIR,
            folder=folder_name,
            block_size=BLOCK_SIZE,
            compression=COMPRESSION
        )

print("\nTẤT CẢ CÁC MODULE ĐÃ ĐƯỢC CHUYỂN ĐỔI THÀNH CÔNG!")

ĐANG XỬ LÝ MODULE: [ICU]

- Đang xử lý: chartevents.csv.gz
  + Nguồn: /content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/chartevents.csv.gz
  + Đích:  /content/drive/MyDrive/NCKH-DDU1231/outputs_parquet/icu/chartevents.parquet
   -> [chartevents.csv.gz] Đã ghi batch 1...
   -> [chartevents.csv.gz] Đã ghi batch 2...
   -> [chartevents.csv.gz] Đã ghi batch 3...
   -> [chartevents.csv.gz] Đã ghi batch 4...
   -> [chartevents.csv.gz] Đã ghi batch 5...
   -> [chartevents.csv.gz] Đã ghi batch 6...
   -> [chartevents.csv.gz] Đã ghi batch 7...
   -> [chartevents.csv.gz] Đã ghi batch 8...
   -> [chartevents.csv.gz] Đã ghi batch 9...
   -> [chartevents.csv.gz] Đã ghi batch 10...
   -> [chartevents.csv.gz] Đã ghi batch 11...
   -> [chartevents.csv.gz] Đã ghi batch 12...
   -> [chartevents.csv.gz] Đã ghi batch 13...
   -> [chartevents.csv.gz] Đã ghi batch 14...
   -> [chartevents.csv.gz] Đã ghi batch 15...
   -> [chartevents.csv.gz] Đã ghi batch 16...
   -> [chartevents.csv.gz] Đã g

In [11]:
# Đường dẫn gốc tới thư mục chứa dữ liệu Parquet (ví dụ)
DATA_DIR = '/content/drive/MyDrive/NCKH-DDU1231/outputs_parquet'

def load(folder, name, columns=None):
    """
    Nạp dữ liệu từ file Parquet.

    :param folder: Tên module (vd: 'icu', 'hosp', 'ed')
    :param name: Tên file (vd: 'chartevents.parquet' hoặc 'chartevents')
    :param columns: Danh sách các cột cần lấy (mặc định None - lấy tất cả).
                    Đọc cột chọn lọc với Parquet giúp tiết kiệm đáng kể RAM.
    """
    path = os.path.join(DATA_DIR, folder, name)

    if os.path.exists(path):
        # pd.read_parquet hỗ trợ đọc chỉ các cột cần thiết qua tham số columns
        df = pd.read_parquet(path, columns=columns)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY (Đường dẫn: {path})")
        return pd.DataFrame()

## **2. Tiền xử lý dữ liệu trước khi chia huấn luyện**

### **2.1. Xây dựng tập cohort**

Cohort được xây dựng bám sát theo các tiêu chí như sau:
- Phải là người trưởng thành (anchor_age > 18)
- Thời gian nằm ICU phải trên 24h (LOS >= 24)
- Chỉ lấy lần nhập ICU đầu tiên trong một lần nhập viện
- Loại các bệnh nhân đã tử vong, xuât hoặc chuyển viện trong vòng 24h đầu

In [13]:
def build_and_val_icu_cohort(patients_df: pd.DataFrame, admissions_df: pd.DataFrame, icustays_df: pd.DataFrame) -> pd.DataFrame:
  # Chuyển đổi định dạng mốc thời gian sang Datetime
  time_cols = {
      'icustays_df': ['intime', 'outtime'],
      'admissions_df': ['admittime', 'dischtime', 'edregtime', 'edouttime'],
      'patients_df' : []
  }

  for col in time_cols['icustays_df']:
        icustays_df[col] = pd.to_datetime(icustays_df[col])
  for col in time_cols['admissions_df']:
        admissions_df[col] = pd.to_datetime(admissions_df[col])
  # 1. Inner Join với patients
  cohort = icustays_df.merge(
  admissions_df[["subject_id", "hadm_id", "admittime", "dischtime", "admission_type",
                        "admission_location", "deathtime",
                        "hospital_expire_flag"]],
          on=['subject_id', 'hadm_id'],
          how='inner'
      )
  # 2. Inner join với admission
  cohort = cohort.merge(
          patients_df[["subject_id", "gender", "anchor_age"]],
          on='subject_id',
          how='inner'
      )

  initial_count = len(cohort)
  print(f"Tổng số lượt ICU ban đầu: {initial_count:,}")

  # Áp dụng các tiêu chí lọc
  # ---------------------------------------------
  # Tiêu chí 1: Người trưởng thành (>= 18 tuổi)
  # ----------------------------------------------
  cohort = cohort[cohort["anchor_age"] >= 18]
  print(f"-> Sau khi lọc người trưởng thành (>=18t): {len(cohort):,} ca")

  # ---------------------------------------------
  # Tiêu chí 2: Lấy ca ICU đầu tiên của mỗi bệnh nhân (First ICU stay per patient)
  # Sắp xếp theo intime để chắc chắn lấy ca đầu tiên trong đời/lịch sử của bệnh nhân
  # ----------------------------------------------
  cohort = (
      cohort.sort_values(["subject_id", "intime"])
            .groupby("subject_id", as_index=False)
            .first()
  )

  print(f"-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: {len(cohort):,} ca")

  # Tính thời gian nằm ICU (tính theo ngày)
  cohort["icu_los_days"] = (
      pd.to_datetime(cohort["outtime"]) - pd.to_datetime(cohort["intime"])
  ).dt.total_seconds() / 86400
  cohort["icu_los_hours"] = cohort["icu_los_days"] * 24

  # ---------------------------------------------
  # Tiêu chí 3: ICU stay trên 1 ngày (>= 1 ngày, tức >= 24 giờ)
  # ---------------------------------------------
  cohort = cohort[cohort["icu_los_days"] >= 1.0]
  print(f"-> Sau khi lọc ICU stay > 1 ngày: {len(cohort):,} ca")

  # Tạo mốc thời gian kết thúc cửa sổ quan sát (intime + 24h)
  cohort['obs_end_time'] = cohort['intime'] + pd.Timedelta(hours=24)

  # Xây dựng nhóm tuổi
  cohort["age_group"] = pd.cut(
      cohort["anchor_age"],
      bins=[17, 30, 50, 65, 80, 120],
      labels=["18-30", "31-50", "51-65", "66-80", "80+"]
  )

  print("=== BẮT ĐẦU THỰC HIỆN Kiểm tra logic ===")
  """
  Tiêu chí loại bỏ các lượt ICU vi phạm logic:
  - Tiêu chí 1: Các khóa phải là duy nhất
  - Tiêu chí 2: Kiểm tra thứ tự các môc thời gian logic
    - Thời gian nhập viện <= Thời gian vào ICU < Thời gian vào ICU + 24h <= Thời gian ra ICU <= Thời gian ra viện
  - Tiêu chí 3: Kiểm tra tính hợp lệ của Nhãn Mục tiêu
    - Thời điểm tử vong của bệnh nhân phải lớn hơn thười gian vào ICU + 24h

  """

  # List lưu giữ các stay_id vi phạm cần loại bỏ
  invalid_stay_ids = set()

  # Check 1: Kiểm tra tính duy nhất của Khóa
  total_rows = len(cohort)
  unique_stays = cohort['stay_id'].nunique()
  print(f"Tổng số dòng: {total_rows} | Số stay_id duy nhất: {unique_stays}")
  if total_rows != unique_stays:
      print("Bị trùng lặp stay_id do phép Join! Cần loại bỏ bản ghi trùng.")
      cohort = cohort.drop_duplicates(subset=['stay_id'])

  # Check 2: Kiểm tra thứ tự mốc thời gian logic
  invalid_time_mask = (
        (cohort['admittime'] > cohort['intime']) |
        (cohort['intime'] >= cohort['outtime']) |
        (cohort['outtime'] > cohort['dischtime'])
    )
  time_faulty_ids = cohort[invalid_time_mask]['stay_id'].tolist()
  invalid_stay_ids.update(time_faulty_ids)
  print(f"Phát hiện {len(time_faulty_ids)} lượt ICU vi phạm thứ tự thời gian sinh lý.")

  # Check 3: Kiểm tra tính hợp lệ của Nhãn Mục tiêu
  early_death_mask = (
        (cohort['hospital_expire_flag'] == 1) &
        (cohort['deathtime'].notna()) &
        (cohort['deathtime'] <= cohort['obs_end_time'])
    )
  early_death_ids = cohort[early_death_mask]['stay_id'].tolist()
  invalid_stay_ids.update(early_death_ids)
  print(f"Phát hiện {len(early_death_ids)} bệnh nhân tử vong TRƯỚC/TRONG 24h đầu ICU.")

  # Loai bỏ các bản ghi vi phạm khỏi Cohort chính thức
  clean_cohort = cohort[~cohort['stay_id'].isin(invalid_stay_ids)].reset_index(drop=True)

  print(f"=== KHỞI TẠO COHORT THÀNH CÔNG ===")
  print(f"Số lượng bệnh nhân hợp lệ cuối cùng: {len(clean_cohort)}")
  print(f"Thời gian nằm ICU trung bình: {cohort['icu_los_days'].mean():.2f} ngày")
  print(f"Tỷ lệ tử vong (Mortality Rate): {clean_cohort['hospital_expire_flag'].mean():.2%}")

  return clean_cohort


In [15]:
print("Loading core tables...")
patients   = load("hosp", "patients.parquet")      # Thông tin bệnh nhân
admissions = load("hosp", "admissions.parquet")    # Thông tin nhập viện
icustays   = load("icu",  "icustays.parquet")      # Thông tin lần nằm

cohort = build_and_val_icu_cohort(patients, admissions, icustays)

Loading core tables...
  ✓ hosp/patients.parquet: 364,627 rows × 3 cols
  ✓ hosp/admissions.parquet: 546,028 rows × 16 cols
  ✓ icu/icustays.parquet: 94,458 rows × 8 cols
Tổng số lượt ICU ban đầu: 94,458
-> Sau khi lọc người trưởng thành (>=18t): 94,458 ca
-> Sau khi chỉ lấy ca ICU đầu tiên của mỗi bệnh nhân: 65,366 ca
-> Sau khi lọc ICU stay > 1 ngày: 51,839 ca
=== BẮT ĐẦU THỰC HIỆN Kiểm tra logic ===
Tổng số dòng: 51839 | Số stay_id duy nhất: 51839
Phát hiện 8714 lượt ICU vi phạm thứ tự thời gian sinh lý.
Phát hiện 171 bệnh nhân tử vong TRƯỚC/TRONG 24h đầu ICU.
=== KHỞI TẠO COHORT THÀNH CÔNG ===
Số lượng bệnh nhân hợp lệ cuối cùng: 43124
Thời gian nằm ICU trung bình: 4.26 ngày
Tỷ lệ tử vong (Mortality Rate): 4.39%


In [16]:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,admission_type,admission_location,deathtime,hospital_expire_flag,gender,anchor_age,icu_los_days,icu_los_hours,obs_end_time,age_group
0,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,2150-11-02 18:02:00,2150-11-12 13:45:00,EW EMER.,EMERGENCY ROOM,NaT,0,F,86,3.893252,93.438056,2150-11-03 19:37:00,80+
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,2157-11-18 22:56:00,2157-11-25 18:00:00,EW EMER.,EMERGENCY ROOM,NaT,0,F,55,1.118032,26.832778,2157-11-21 19:18:02,51-65
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,2110-04-11 15:08:00,2110-04-14 15:00:00,EW EMER.,PACU,NaT,0,F,46,1.338588,32.126111,2110-04-12 15:52:22,31-50
3,10002013,23581541,39060235,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2160-05-18 10:00:53,2160-05-19 17:33:33,1.314352,2160-05-18 07:45:00,2160-05-23 13:30:00,SURGICAL SAME DAY ADMISSION,PHYSICIAN REFERRAL,NaT,0,F,53,1.314352,31.544444,2160-05-19 10:00:53,51-65
4,10002114,27793700,34672098,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2162-02-17 23:30:00,2162-02-20 21:16:27,2.907257,2162-02-17 22:32:00,2162-03-04 15:16:00,OBSERVATION ADMIT,PHYSICIAN REFERRAL,NaT,0,M,56,2.907257,69.774167,2162-02-18 23:30:00,51-65


In [17]:
cohort.isna().sum()

,0
subject_id,0
hadm_id,0
stay_id,0
first_careunit,0
last_careunit,0
intime,0
outtime,0
los,0
admittime,0
dischtime,0


**Có missing cần lưu ý**
- `deathtime`: Có thể `Null - None` do bệnh nhân không tử vong

In [18]:
cohort.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'admittime', 'dischtime', 'admission_type',
       'admission_location', 'deathtime', 'hospital_expire_flag', 'gender',
       'anchor_age', 'icu_los_days', 'icu_los_hours', 'obs_end_time',
       'age_group'],
      dtype='object')

### **2.2. Chia tập train - test - validate để huấn luyện mô hình & Xác định nhãn cho bài toán**

- Thực hiện chia tập dữ liệu với tỷ lệ tử vong ở mỗi tập là như nhau
  - Chia theo tý lệ train (70%) - test (20%) - val (10%)
- **Nhãn mục tiêu** `hospital_expire_flag`
  - Với `0` là bệnh nhân sống sót và `1` là bệnh nhân tử vong

In [19]:
# Sinh x và y
X = cohort.drop(columns=['hospital_expire_flag'])
y = cohort['hospital_expire_flag']

In [20]:
from sklearn.model_selection import train_test_split

def split_data(
    X: pd.DataFrame,
    y: pd.Series,
    target_col: str = 'hospital_expire_flag',
    test_size: float = 0.2,
    val_size: float = 0.125,
    random_state: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Chia dataset thành 3 tập Train/Val/Test theo `subject_id` để chống Data Leakage.
    Tỷ lệ mặc định: 70% Train - 20% Val - 10% Test.
    """
    # Tách 20% cho tập Test (Còn lại 80% cho Train + Val)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # Chia 80% đó thành Train (70% tổng) và Val (10% tổng)
    # Tỷ lệ tập Val trong tập Train_Val là: 10% / 80% = 0.125 (12.5%)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=val_size,
        stratify=y_train_val,
        random_state=random_state
    )

    # 4. Kiểm tra kích thước các tập
    print(f"Kích thước tập Train: {X_train.shape[0]} ({len(X_train)/len(cohort):.0%})")
    print(f"Kích thước tập Val:   {X_val.shape[0]} ({len(X_val)/len(cohort):.0%})")
    print(f"Kích thước tập Test:  {X_test.shape[0]} ({len(X_test)/len(cohort):.0%})")

    # 5. Kiểm tra tỷ lệ biến mục tiêu có được giữ nguyên không
    print("\nTỷ lệ nhãn trong tập Train:\n", y_train.value_counts(normalize=True))
    print("\nTỷ lệ nhãn trong tập Val:\n", y_val.value_counts(normalize=True))
    print("\nTỷ lệ nhãn trong tập Test:\n", y_test.value_counts(normalize=True))

    return X_train, X_test, y_train_val, y_test, X_train, X_val, y_train, y_val

In [21]:
X_train, X_test, y_train_val, y_test, X_train, X_val, y_train, y_val = split_data(X, y)

Kích thước tập Train: 30186 (70%)
Kích thước tập Val:   4313 (10%)
Kích thước tập Test:  8625 (20%)

Tỷ lệ nhãn trong tập Train:
 hospital_expire_flag
0    0.956105
1    0.043895
Name: proportion, dtype: float64

Tỷ lệ nhãn trong tập Val:
 hospital_expire_flag
0    0.956179
1    0.043821
Name: proportion, dtype: float64

Tỷ lệ nhãn trong tập Test:
 hospital_expire_flag
0    0.956174
1    0.043826
Name: proportion, dtype: float64


## **3. Xây dựng các bảng đặc trưng thực hiện tiền xử lý cho dữ liệu huấn luyện**

In [5]:
from typing import Dict, Tuple, List
import pyarrow as pa
import pyarrow.parquet as pq

### **3.1. Xây dựng bảng chart vitals từ `chartevent`**

Kết hợp với bảng cohort với cửa số được lấy trong vòng 24h khi nằm ở icu
- intime <= charttime <= intime + 24h
- Lọc các chỉ số theo mã id hợp lệ rồi học các giá trị ngoại lai
- Thực hiện sinh thêm các đặc trưng mới với các chỉ số phức tạp

In [6]:
def extract_clean_chartevents(
    events_df: pd.DataFrame,
    cohort_df: pd.DataFrame,
    item_mapping: Dict[int, str],
    outlier_bounds: Dict[str, Tuple[float, float]]
) -> pd.DataFrame:
    """
    Lọc dữ liệu chartevents theo stay_id trong 24h ICU và làm sạch nhiễu sinh lý.
    """
    if events_df.empty or cohort_df.empty:
        return pd.DataFrame()

    # Lọc các itemid hợp lệ
    df = events_df[events_df['itemid'].isin(item_mapping.keys())].copy()
    df['feature_name'] = df['itemid'].map(item_mapping)

    # Merge lấy intime từ cohort để khóa cửa sổ 24h
    df = df.merge(
        cohort_df[['stay_id', 'intime']],
        on='stay_id',
        how='inner'
    )

    # Lọc thời gian: 0h <= delta_hours <= 24h
    df['charttime'] = pd.to_datetime(df['charttime'])
    df['intime'] = pd.to_datetime(df['intime'])
    df['delta_hours'] = (df['charttime'] - df['intime']).dt.total_seconds() / 3600.0

    mask_time = (df['delta_hours'] >= 0.0) & (df['delta_hours'] <= 24.0)
    df = df[mask_time].copy()

    # Chuyển đổi dữ liệu đo về dạng số
    df['valuenum'] = pd.to_numeric(df['valuenum'], errors='coerce')
    df = df.dropna(subset=['valuenum']).copy()

    # Quy đổi đơn vị đo chuẩn (F -> C, lbs -> kg, inches -> cm)
    if 'valueuom' in df.columns:
        uom_str = df['valueuom'].astype(str).str.lower()

        # Độ F -> C
        f_mask = uom_str.str.contains('f', na=False) | (
            df['feature_name'].str.contains('temp', case=False) & (df['valuenum'] > 50.0)
        )
        df.loc[f_mask, 'valuenum'] = (df.loc[f_mask, 'valuenum'] - 32.0) * 5.0 / 9.0

        # lbs -> kg
        lbs_mask = uom_str.str.contains('lb', na=False)
        df.loc[lbs_mask, 'valuenum'] = df.loc[lbs_mask, 'valuenum'] * 0.453592

        # inches -> cm
        inch_mask = uom_str.str.contains('inch|in', na=False)
        df.loc[inch_mask, 'valuenum'] = df.loc[inch_mask, 'valuenum'] * 2.54

    # Lọc nhiễu sinh lý theo Outlier Bounds
    for feat_name, (min_val, max_val) in outlier_bounds.items():
        feat_mask = df['feature_name'] == feat_name
        invalid_mask = feat_mask & ((df['valuenum'] < min_val) | (df['valuenum'] > max_val))
        df.loc[invalid_mask, 'valuenum'] = np.nan

    df = df.dropna(subset=['valuenum']).copy()

    # Sắp xếp lại dữ liệu theo ID và mốc thời gian
    cols_to_keep = ['stay_id', 'charttime', 'delta_hours', 'feature_name', 'valuenum']
    df = df.sort_values(by=['stay_id', 'feature_name', 'charttime'])[cols_to_keep]

    return df.reset_index(drop=True)

In [2]:
# Định nghĩa mapping itemid
item_mapping = {
    # Sinh hiệu cơ bản & Hô hấp
    220045: "heart_rate",        # Nhịp tim (bpm)
    220210: "resp_rate",         # Nhịp thở (lần/phút)
    223762: "temp_c",            # Nhiệt độ độ C (°C)
    223761: "temp_f",            # Nhiệt độ độ F (°F)
    220277: "spo2",              # Nồng độ Oxy máu SpO2 (%)
    223835: "fio2",              # Tỷ lệ Oxy hít vào FiO2 (%)

    # Huyết áp xâm lấn (Arterial Line)
    220050: "sbp_line",          # Huyết áp tâm thu xâm lấn (mmHg)
    220051: "dbp_line",          # Huyết áp tâm trương xâm lấn (mmHg)
    220052: "mbp_line",          # Huyết áp trung bình xâm lấn (mmHg)

    # Huyết áp không xâm lấn (NIBP) - Giữ lại để tránh thiếu dữ liệu
    220179: "sbp_nibp",          # Huyết áp tâm thu NIBP (mmHg)
    220180: "dbp_nibp",          # Huyết áp tâm trương NIBP (mmHg)
    220181: "mbp_nibp",          # Huyết áp trung bình NIBP (mmHg)

    # Đường huyết
    225664: "glucose",           # Glucose máu (mg/dL)

    # Khí máu động mạch (Arterial Blood Gas)
    223830: "ph",                # pH máu động mạch
    220224: "pao2",              # Áp suất một phần Oxy PaO2 (mmHg)
    220235: "paco2",             # Áp suất một phần CO2 PaCO2 (mmHg)
    224828: "base_excess",       # Kiềm dư Base Excess (mEq/L)
    220227: "sao2",              # Độ bão hòa Oxy động mạch SaO2 (%)

    # Công thức máu (CBC)
    220228: "hemoglobin",        # Hemoglobin (g/dL)
    220545: "hematocrit",        # Hematocrit (%)
    220546: "wbc",               # Bạch cầu WBC (K/uL)
    227457: "platelet",          # Tiểu cầu Platelets (K/uL)

    # Chức năng thận
    220615: "creatinine",        # Creatinine (mg/dL)
    225624: "bun",               # Blood Urea Nitrogen (mg/dL)

    # Chức năng gan
    220644: "alt",               # Alanine Aminotransferase ALT (U/L)
    220587: "ast",               # Aspartate Aminotransferase AST (U/L)
    225690: "bilirubin_total",   # Bilirubin toàn phần (mg/dL)
    227456: "albumin",           # Albumin (g/dL)
    225612: "alp",               # Alkaline Phosphatase (U/L)

    # Thang điểm hôn mê Glasgow (GCS)
    223900: "gcs_verbal",        # GCS - Đáp ứng lời nói (1 - 5)
    223901: "gcs_motor",         # GCS - Đáp ứng vận động (1 - 6)
    220739: "gcs_eye",           # GCS - Mở mắt (1 - 4)

    # Nội tiết
    228236: "insulin",           # Insulin (uIU/mL)
    227463: "cortisol",          # Cortisol (ug/dL)

    # Trọng lượng & Chiều cao
    224639: "weight_daily",      # Cân nặng hàng ngày (kg)
    226512: "weight_admit",      # Cân nặng khi nhập viện (kg)
    226707: "height_inch",       # Chiều cao (Inches)
    226730: "height_cm"          # Chiều cao (cm)
}

# Định nghĩa khoảng outlier
outlier_bounds = {
    # Sinh hiệu & Hô hấp
    "heart_rate": (20.0, 250.0),
    "resp_rate": (4.0, 60.0),
    "temp_c": (25.0, 43.0),
    "temp_f": (77.0, 109.4),
    "spo2": (50.0, 100.0),
    "fio2": (21.0, 100.0),        # Trong chartevents, FiO2 thường lưu dạng % (21-100)

    # Huyết áp
    "sbp_line": (40.0, 280.0),
    "dbp_line": (20.0, 150.0),
    "mbp_line": (20.0, 200.0),
    "sbp_nibp": (40.0, 280.0),
    "dbp_nibp": (20.0, 150.0),
    "mbp_nibp": (20.0, 200.0),

    # Đường huyết & Khí máu
    "glucose": (10.0, 2000.0),
    "ph": (6.5, 7.8),
    "pao2": (20.0, 700.0),
    "paco2": (10.0, 150.0),
    "base_excess": (-30.0, 30.0),
    "sao2": (50.0, 100.0),

    # Huyết học (CBC)
    "hemoglobin": (2.0, 25.0),
    "hematocrit": (5.0, 75.0),
    "wbc": (0.1, 200.0),
    "platelet": (5.0, 2000.0),

    # Chức năng thận
    "creatinine": (0.1, 30.0),
    "bun": (1.0, 250.0),

    # Chức năng gan
    "alt": (1.0, 10000.0),
    "ast": (1.0, 10000.0),
    "bilirubin_total": (0.1, 80.0),
    "albumin": (0.5, 7.0),
    "alp": (5.0, 5000.0),

    # GCS
    "gcs_verbal": (1.0, 5.0),
    "gcs_motor": (1.0, 6.0),
    "gcs_eye": (1.0, 4.0),

    # Nội tiết
    "insulin": (0.1, 500.0),
    "cortisol": (0.1, 200.0),

    # Thể trạng
    "weight_daily": (20.0, 300.0),
    "weight_admit": (20.0, 300.0),
    "height_inch": (20.0, 100.0),
    "height_cm": (50.0, 250.0)
}

In [11]:
def process_chartevents_in_chunks(
    input_parquet_path: str,
    output_parquet_path: str,
    cohort_df: pd.DataFrame,
    batch_size: int = 1_000_000
):
    """
    Đọc chartevents.parquet theo từng batch nhỏ, làm sạch và ghi đệm ngay xuống đĩa.
    Giúp duy trì bộ nhớ RAM luôn ở mức cực thấp (< 2-3GB).
    """
    if not os.path.exists(input_parquet_path):
        print(f"❌ Không tìm thấy file: {input_parquet_path}")
        return

    target_itemids = list(ITEM_MAPPING.keys())
    cols_to_read = ['stay_id', 'charttime', 'itemid', 'valuenum', 'valueuom']

    # 1. Mở file Parquet dưới dạng Stream (chưa load vào RAM)
    parquet_file = pq.ParquetFile(input_parquet_path)
    print(f"📊 Tổng số dòng trong file gốc: {parquet_file.metadata.num_rows:,} dòng.")
    print(f"⚡ Bắt đầu stream từng batch {batch_size:,} dòng...")

    writer = None
    total_processed_rows = 0
    total_clean_rows = 0

    # 2. Duyệt qua từng batch dữ liệu
    for batch_idx, record_batch in enumerate(
        parquet_file.iter_batches(batch_size=batch_size, columns=cols_to_read)
    ):
        # Chuyển batch hiện tại thành Pandas DataFrame
        df_chunk = record_batch.to_pandas()
        total_processed_rows += len(df_chunk)

        # Lọc nhanh các itemid mục tiêu trên RAM
        df_chunk = df_chunk[df_chunk['itemid'].isin(target_itemids)]

        if not df_chunk.empty:
            # 3. Áp dụng logic làm sạch 24h ICU & Outliers cho từng chunk
            df_clean_chunk = extract_clean_chartevents(
                events_df=df_chunk,
                cohort_df=cohort_df,
                item_mapping=ITEM_MAPPING,
                outlier_bounds=OUTLIER_BOUNDS
            )

            if not df_clean_chunk.empty:
                total_clean_rows += len(df_clean_chunk)
                table_chunk = pa.Table.from_pandas(df_clean_chunk)

                # Khởi tạo Writer ở batch đầu tiên thu được dữ liệu sạch
                if writer is None:
                    writer = pq.ParquetWriter(
                        output_parquet_path,
                        table_chunk.schema,
                        compression='snappy'
                    )

                # Ghi trực tiếp xuống đĩa và giải phóng RAM ngay
                writer.write_table(table_chunk)

        print(f"   -> Đã quét {total_processed_rows:,} dòng | Thu được {total_clean_rows:,} dòng sạch...")

        # 4. Ép Python thu gom rác để giải phóng RAM tối đa
        del df_chunk
        gc.collect()

    # Đóng file writer sau khi hoàn tất
    if writer:
        writer.close()
        print(f"\n✅ HOÀN THÀNH!")
        print(f"📁 Đã ghi tổng cộng {total_clean_rows:,} dòng sạch vào: {output_parquet_path}")
    else:
        print("\n⚠️ Không có dữ liệu nào thỏa mãn điều kiện lọc.")

In [ ]:
if __name__ == "__main__":

    # 2. Đường dẫn file nguồn và đệm đĩa file sạch
    chartevents_in = os.path.join(BASE_OUTPUT_DIR, 'icu', 'chartevents.parquet')
    chartevents_out = os.path.join(BASE_OUTPUT_DIR, 'icu', 'vital_signs_24h.parquet')

    # 3. Chạy tiến trình đọc - ghi theo luồng (Cố định batch_size = 1,000,000 dòng)
    process_chartevents_in_chunks(
        input_parquet_path=chartevents_in,
        output_parquet_path=chartevents_out,
        cohort_df=X_train,
        batch_size=1_000_000
    )

### **3.2. Xây dựng bảng lab test từ bảng `labevents`**

### **3.3. Xây dựng các bảng còn lại phụ trợ cho bảng chính**

## **4. Tiền xử lý dữ liệu**

## **5. Huấn luyện mô hình học máy**